# Notebook 4 — Structured Streaming simulation

Chunks processed events into time-ordered JSON under `streaming/input/`, runs a watermark + sliding window aggregation, writes append JSON to `streaming/output/`.


In [ ]:
import os
import sys
import time

sys.path.insert(0, os.path.abspath("."))
import hdfs_paths as hp

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, window, date_trunc

spark = (
    SparkSession.builder.appName("MusicTrend_04_Streaming")
    .config("spark.sql.shuffle.partitions", hp.SHUFFLE_PARTITIONS)
    .getOrCreate()
)


In [ ]:
# Materialize hourly JSON chunks for file source
events = spark.read.parquet(hp.PROCESSED_EVENTS)
(
    events.withColumn("hour_chunk", date_trunc("hour", col("event_timestamp")))
    .write.mode("overwrite")
    .partitionBy("hour_chunk")
    .json(hp.STREAMING_INPUT)
)
print("Wrote streaming input JSON to", hp.STREAMING_INPUT)


In [ ]:
from pyspark.sql.types import StructType, StringType, TimestampType

event_schema = (
    StructType()
    .add("user_id", StringType())
    .add("artist_id", StringType())
    .add("event_timestamp", TimestampType())
)

streaming_events = (
    spark.readStream.schema(event_schema)
    .option("maxFilesPerTrigger", 1)
    .json(hp.STREAMING_INPUT)
)

momentum_stream = (
    streaming_events.withWatermark("event_timestamp", "1 hour")
    .groupBy(
        window(col("event_timestamp"), "7 days", "1 day"),
        col("artist_id"),
    )
    .agg(count("*").alias("play_count_7d"))
)

query = (
    momentum_stream.writeStream.outputMode("append")
    .format("json")
    .option("path", hp.STREAMING_OUTPUT)
    .option("checkpointLocation", hp.STREAMING_CHECKPOINT)
    .trigger(processingTime="30 seconds")
    .start()
)

# Run briefly to emit batches (adjust sleep for cluster)
time.sleep(120)
query.stop()
print("Streaming query stopped.")


In [ ]:
out = spark.read.json(hp.STREAMING_OUTPUT)
out.orderBy(col("play_count_7d").desc()).show(20)


## Production Kafka source

The same `groupBy`/watermark logic applies; only the reader changes, e.g.:

```python
spark.readStream.format("kafka")
  .option("kafka.bootstrap.servers", "host:9092")
  .option("subscribe", "listen-events")
  ...
```

Partitioning by `artist_id` in Kafka would preserve key locality for aggregations.


## Outputs confirmed

- Checkpoint + append JSON under `streaming/output/`.
- Sample top artists by rolling play count shown.
